# Implementing an LR-Table-Generator

## A Grammar for Grammars

To generate LR(1) tables, we reuse the typed grammar parser and FIRST-set computation from the SLR notebook.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
printfn "%s" (System.IO.File.ReadAllText "Examples/arith.g")

The grammar-for-grammars lexer and parser are compiled with FsLexYacc in `../Chapter-08/GrammarSyntax`.

In [ ]:
#load "SLR.fsx"
open SLR
open GrammarSyntax

type ExtendedMarkedRule = { Core: MarkedRule; Follow: Set<string> }

let combineRules items =
    items |> Seq.groupBy (fun item -> item.Core)
    |> Seq.map (fun (core, group) -> { Core = core; Follow = group |> Seq.collect (fun item -> item.Follow) |> Set.ofSeq })
    |> Set.ofSeq
let isComplete (item: ExtendedMarkedRule) = SLR.isComplete item.Core
let symbolAfterDot (item: ExtendedMarkedRule) = SLR.symbolAfterDot item.Core
let nextVar item = SLR.nextVar item.Core
let moveDot (item: ExtendedMarkedRule) = { item with Core = SLR.moveDot item.Core }
let toRule (item: ExtendedMarkedRule) = item.Core.Rule
let toMarkedRule (item: ExtendedMarkedRule) = item.Core

We start by generating both scanner and parser.

In [ ]:
let parse = Parse.file

In [ ]:
let grammar = parse "Examples/arith.g"
printfn "%A" grammar

## Grammar Rules

Immutable `Rule` records use F# structural equality and ordering. Terminal and nonterminal union cases remove the need to infer their role from spelling.

`Parse.file` already constructs typed grammar rules.

The typed `Symbol` distinguishes terminals from nonterminals.

Given a list `Rules` of `GrammarRules`, the function `variables(Rules)` returns the set of all *variables* occuring in `Rules`.

Given a set `Rules` of `GrammarRules`, the function `terminals(Rules)` returns the set of all *tokens* and *literals* occuring in `Rules`.

## Extended Marked Rules

The type `ExtendedMarkedRule` represents the marked rule
$$ v \rightarrow \alpha \bullet \beta : L $$

`Rule.Head` stores $v$, `Rule.Body` stores the entire right-hand side $\alpha\beta$, and `Dot` stores the length of $\alpha$. Each symbol is a typed terminal or nonterminal. Immutable F# records provide structural equality and comparison, so marked rules can be elements of sets.

`Core` stores the marked rule and `Follow` stores the LR(1) lookahead set $L$.

Given an *extended marked rule* `item`, the function `isComplete` checks, whether the *extended marked rule* `item` has the form
$$ c \rightarrow \alpha\; \bullet: L,$$
i.e. it checks, whether the $\bullet$ is at the end of the grammar rule.

Given an *extended marked rule* `item` of the form
$$ c \rightarrow \alpha \bullet X\, \delta: L, $$
the function `symbolAfterDot` returns the *symbol* $X$. If there is no symbol after the $\bullet$, the method returns `None`.

Given an extended marked rule, this function returns the variable following the dot.  If there is no variable following the dot, the function returns `None`.

The function `moveDot item` transforms an *extended marked rule*  of the form 
$$ c \rightarrow \alpha \bullet X\, \beta: L $$
into an *extended marked rule* of the form
$$ c \rightarrow \alpha\, X \bullet \beta: L, $$
i.e. the $\bullet$ is moved over the next symbol.  Invocation of this method assumes that there is a symbol
following the $\bullet$.

The function `toRule item` turns the *extended marked rule* `item` into  a `Rule`, i.e. the *extended marked rule*
$$ c \rightarrow \alpha \bullet \beta: L $$
is turned into the grammar rule
$$ c \rightarrow \alpha\, \beta. $$

The function `toRule item` turns the *extended marked rule* `item` into a `MarkedRule`, i.e. the *extended marked rule*
$$ c \rightarrow \alpha \bullet \beta: L $$
is turned into the marked rule
$$ c \rightarrow \alpha\bullet \beta. $$

`MarkedRule` from SLR is the core of an extended marked rule.

Given a set of *extended marked rules* `M`, the function `combineRules` *combines* those extended marked ruless that have the same *core*:  If 
$$ a \rightarrow \beta \bullet \gamma : L $$
is an extended marked rule, then its *core* is defined as the marked rule
$$ a \rightarrow \beta \bullet \gamma. $$
If $a \rightarrow \beta \bullet \gamma : L_1$ and $a \rightarrow \beta \bullet \gamma : L_2$ are two extended marked rules, then they can be *combined* into the rule
$$ a \rightarrow \beta \bullet \gamma : L_1\cup L_2 $$

In [ ]:
let combined = combineRules []
printfn "%A" combined

## LR-Table-Generation

The grammar is immutable. States and tables are returned as separate values by `computeTables`.

The FIRST map is reused from SLR.

The LR implementation computes FIRST sets, closure, canonical LR(1) states, action sets and gotos. It keeps items with different lookahead sets distinct across states.

Reduction actions contain the actual typed rule.

`computeFirst grammar` computes the FIRST map by fixed-point iteration. We include $\lambda$ exactly when a nonterminal derives the empty word:
$$\mathrm{First}(a)=\{t\in T\mid\exists\gamma:\ a\Rightarrow^*t\gamma\}\ \cup\ \{\lambda\mid a\Rightarrow^*\lambda\}.$$
For a list of grammar symbols:
- $\mathrm{FirstList}(\lambda)=\{\lambda\}$.
- $\mathrm{FirstList}(t\beta)=\{t\}$ for terminal $t$.
- $\mathrm{FirstList}(a\beta)=\mathrm{First}(a)$ if $\lambda\notin\mathrm{First}(a)$.
- Otherwise $\mathrm{FirstList}(a\beta)=(\mathrm{First}(a)\setminus\{\lambda\})\cup\mathrm{FirstList}(\beta)$.
For productions $a\rightarrow\alpha_1\mid\cdots\mid\alpha_n$ we repeatedly enlarge
$$\mathrm{First}(a)=\bigcup_{i=1}^{n}\mathrm{FirstList}(\alpha_i).$$
The empty string `""` represents $\lambda$ in the implementation.

FIRST sets and epsilon-aware `firstList` are the identical functions taught in the SLR notebook and imported above. They are reused without a second implementation.

Given a list of grammar symbols `alpha`, the function `firstList first alpha` computes the function $\texttt{FirstList}(\alpha)$ that has been defined above.  If `alpha` is *nullable*, then the result will contain the empty string $\varepsilon = \texttt{''}$.

The arguments `S` and `T` of `epsUnion` are sets that contain tokens and, additionally, they might contain the empty string.

Unlike SLR, LR(1) reduction tokens come from each completed item’s lookahead set.

If $\mathcal{M}$ is a set of *extended marked rules*, then the *closure* of $\mathcal{M}$ is the smallest set $\mathcal{K}$ such that
we have the following:
- $\mathcal{M} \subseteq \mathcal{K}$,
- If $a \rightarrow \beta \bullet c\, \delta: L$ is a *extended marked rule* from 
  $\mathcal{K}$, $c$ is a variable, and $t\in L$ and if, furthermore,
  $c \rightarrow \gamma$ is a grammar rule,
  then the marked rule $c \rightarrow \bullet \gamma: \texttt{First}(\delta\,t)$
  is an element of $\mathcal{K}$:
  $$(a \rightarrow \beta \bullet c\, \delta:L) \in \mathcal{K} 
         \;\wedge\; 
         (c \rightarrow \gamma) \in R \;\wedge\; t \in L
         \;\Rightarrow\; (c \rightarrow \bullet \gamma: \texttt{First}(\delta\,t)) \in \mathcal{K}
  $$

We define $\texttt{closure}(\mathcal{M}) := \mathcal{K}$.  The function `closure` computes this closure for a given set of *extended marked rules* via a *fixed-point iteration*.

In [ ]:
let closure grammar first items =
    let mutable result = combineRules items
    let mutable changed = true
    while changed do
        let additions =
            result |> Seq.collect (fun item ->
                match nextVar item with
                | None -> Seq.empty
                | Some name ->
                    let beta = item.Core.Rule.Body |> List.skip (item.Core.Dot + 1)
                    let lookahead =
                        item.Follow |> Seq.collect (fun token -> firstList first (beta @ [Terminal token])) |> Set.ofSeq
                    grammar.Rules |> Seq.filter (fun rule -> rule.Head = name)
                    |> Seq.map (fun rule -> { Core = { Rule = rule; Dot = 0 }; Follow = lookahead }))
        let updated = Seq.append result additions |> combineRules
        changed <- updated <> result
        result <- updated
    result

let goto grammar first items symbol =
    items |> Seq.filter (fun item -> symbolAfterDot item = Some symbol)
    |> Seq.map moveDot |> closure grammar first

Given a set of *extended marked rules* $\mathcal{M}$ and a *grammar symbol* $X$, the function $\texttt{goto}(\mathcal{M}, X)$ 
is defined as follows:
$$\texttt{goto}(\mathcal{M}, X) := \texttt{closure}\Bigl( \bigl\{ 
   a \rightarrow \beta\, X \bullet \delta:L \bigm| (a \rightarrow \beta \bullet X\, \delta:L) \in \mathcal{M} 
   \bigr\} \Bigr).
$$

The function `allStates` computes the set of all states of an *LR-parser*.  The function starts with the state
$$ \texttt{closure}\bigl(\{ \widehat{s} \rightarrow \bullet s : {\$} \}\bigr) $$
and then tries to compute new states by using the function `goto`.  This computation proceeds via a 
*fixed-point iteration*.  Once all states have been computed, the function assigns names to these states.
The array index is the state number; the discovery map assigns the same number to every structurally equal state.

In [ ]:
// Breadth-first discovery and structural ordering produce reproducible state numbers.
let allStates grammar first =
    let augmented = { Head = "ŝ"; Body = [Nonterminal grammar.Start] }
    let start = closure grammar first (Set.singleton { Core = { Rule = augmented; Dot = 0 }; Follow = Set.singleton "$" })
    let states = ResizeArray<Set<ExtendedMarkedRule>>()
    let names = System.Collections.Generic.Dictionary<Set<ExtendedMarkedRule>, int>()
    let mutable transitions = Map.empty
    states.Add start
    names.Add(start, 0)
    let mutable position = 0
    while position < states.Count do
        let symbols = states[position] |> Seq.choose symbolAfterDot |> Set.ofSeq
        for symbol in symbols do
            let target = goto grammar first states[position] symbol
            let targetName =
                match names.TryGetValue target with
                | true, name -> name
                | _ ->
                    let name = states.Count
                    states.Add target
                    names.Add(target, name)
                    name
            transitions <- Map.add (position, symbol) targetName transitions
        position <- position + 1
    states.ToArray(), transitions

The following function computes the *action table* and is defined as follows:
- If $\mathcal{M}$ contains an *extended marked rule* of the form $a \rightarrow \beta \bullet t\, \delta:L$
  then we have
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{shift}, \texttt{goto}(\mathcal{M},t) \rangle.$$
- If $\mathcal{M}$ contains an *extended marked rule* of the form $a \rightarrow \beta\, \bullet:L$ and we have
  $t \in L$, then we define
  $$\texttt{action}(\mathcal{M},t) := \langle \texttt{reduce}, a \rightarrow \beta \rangle$$
- If $\mathcal{M}$ contains the *extended marked rule* $\widehat{s} \rightarrow s \bullet:\{\$\}$, then we define 
  $$\texttt{action}(\mathcal{M},\$) := \texttt{accept}. $$
- Otherwise, we have
  $$\texttt{action}(\mathcal{M},t) := \texttt{error}. $$

In [ ]:
let computeTables grammar =
    let first = computeFirst grammar
    let states, transitions = allStates grammar first
    let mutable actions = Map.empty
    let mutable gotos = Map.empty
    for KeyValue((state, symbol), target) in transitions do
        match symbol with
        | Terminal token -> actions <- addAction (state, token) (Shift target) actions
        | Nonterminal name -> gotos <- Map.add (state, name) target gotos
    for state in 0 .. states.Length - 1 do
        for item in states[state] do
            if isComplete item then
                if item.Core.Rule.Head = "ŝ" then
                    actions <- addAction (state, "$") Accept actions
                else
                    for token in item.Follow do
                        actions <- addAction (state, token) (Reduce item.Core.Rule) actions
    { States = states; Actions = actions; Gotos = gotos }

let describeItem item = sprintf "%s, %A" (SLR.describeItem item.Core) item.Follow

let analyse file =
    let table = computeTables (Parse.file file)
    printTable describeItem table
    table

The function `computeTables` computes the *goto table*.

In [ ]:
checkNotebook ((conflicts (analyse "Examples/arith.g")).IsEmpty)

In [ ]:
checkNotebook ((conflicts (analyse "Examples/postfix.g")).IsEmpty)

In [ ]:
checkNotebook ((conflicts (analyse "Examples/arith-small.g")).IsEmpty)

In [ ]:
let slr = SLR.computeTables (Parse.file "Examples/c-grammar.g")
let lr = computeTables (Parse.file "Examples/c-grammar.g")
checkNotebook (not ((conflicts slr).IsEmpty))
printfn "C grammar: SLR conflicts %d; LR conflicts %d (including dangling else)" (conflicts slr).Length (conflicts lr).Length
let assignmentGrammar = Parse.text "s: l '=' r | r; l: '*' r | IDENTIFIER; r: l;"
checkNotebook (not ((conflicts (SLR.computeTables assignmentGrammar)).IsEmpty))
checkNotebook ((conflicts (computeTables assignmentGrammar)).IsEmpty)

The mathematical descriptions sometimes write the augmented production as $\widehat{s}\rightarrow s\,\$$. The implementation uses the equivalent convention $\widehat{s}\rightarrow s$, keeping `$` exclusively as the external end-of-input lookahead. The unmodified C fixture remains ambiguous around dangling `else`; its remaining LR conflicts are reported rather than resolved silently.